# Distributional Sentiment V1

Clean research workspace for the paper question:

> Does the distribution of financial-news sentiment contain information beyond mean sentiment?

This notebook is intentionally separate from the student notebooks. It uses the existing LLM-labeled `result.csv` as the v1 sentiment source, treats discrete labels as a baseline measurement choice, and focuses first on future volatility and tail risk.

## Research Design Decisions

- **Sentiment model for v1:** existing LLM classifications in `result.csv`.
- **Baseline encoding:** signed class mapping: A=-1, B=-0.5, C=0, D=0.5, E=1.
- **Core benchmark:** every experiment controls for mean sentiment.
- **Primary endpoints:** future realized volatility and absolute-return tail risk.
- **Secondary endpoints:** returns, reversals, and volume.
- **Main comparison:** baseline model versus distributional-sentiment model.

Later robustness should compare this discrete-label baseline against continuous LLM scores, FinBERT probabilities, and alternative class mappings.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

RESULT_PATH = Path("result.csv")
MARKET_CACHE_PATH = Path("data/market_panel.csv")

START_DATE = "2020-05-01"
END_DATE = "2025-05-01"

# Set True inside the notebook if market_panel.csv is absent and network access is available.
FETCH_MARKET_DATA = False

TAIL_QUANTILE = 0.90
OOS_SPLIT_DATE = "2023-01-01"
PRIMARY_TARGET = "fwd_rv_5d"

## 1. Load And Audit LLM-Labeled Headlines

In [2]:
if not RESULT_PATH.exists():
    raise FileNotFoundError(f"Missing {RESULT_PATH}. Add the LLM-labeled headline file before running this notebook.")

raw = pd.read_csv(RESULT_PATH, low_memory=False)
raw["date"] = pd.to_datetime(raw["date"], utc=True, errors="coerce")
raw["Date"] = pd.to_datetime(raw.get("Date", raw["date"].dt.date), errors="coerce").dt.date

audit = {
    "rows": len(raw),
    "start": raw["date"].min(),
    "end": raw["date"].max(),
    "calendar_days": raw["Date"].nunique(),
    "sources": raw["source"].nunique() if "source" in raw else np.nan,
    "duplicate_headlines": int(raw["headline"].duplicated().sum()) if "headline" in raw else np.nan,
}
display(pd.Series(audit, name="headline_file_audit"))

display(raw["source"].value_counts(dropna=False).rename("source_count"))
display(raw.head())

rows                                      142343
start                  2020-05-01 00:00:09+00:00
end                    2025-04-30 22:59:23+00:00
calendar_days                               1826
sources                                        2
duplicate_headlines                          438
Name: headline_file_audit, dtype: object

source
Guardian    90584
NYTimes     51759
Name: source_count, dtype: int64

,headline,source,date,content,section_name,event,sentiment_class,time_horizon_class,index_relatability_class,surprise_class,day_uncertainty_class,day_disagreement_class,Date
0,U.S. Stocks End the Week Lower After Tech Earn...,NYTimes,2020-05-01 04:01:12+00:00,NaN,Business Day,Overall Dataset,B,A,C,A,C,C,2020-05-01
1,Companies Sell the Blood of Recovered Coronavi...,NYTimes,2020-05-01 04:01:49+00:00,NaN,World,Overall Dataset,B,B,A,B,C,C,2020-05-01
2,Hundreds of Rohingya Refugees Stuck at Sea Wit...,NYTimes,2020-05-01 04:39:21+00:00,NaN,World,Overall Dataset,C,A,A,A,C,C,2020-05-01
3,Racing Against the Virus From Inside Australia...,NYTimes,2020-05-01 05:29:09+00:00,NaN,World,Overall Dataset,C,A,A,A,C,C,2020-05-01
4,"‘Hospital Needs to Be Quarantined,’ but Works ...",NYTimes,2020-05-01 06:33:57+00:00,NaN,World,Overall Dataset,C,A,A,A,C,C,2020-05-01


## 2. Clean Labels And Encode Sentiment

The signed sentiment scale is the main v1 object because it makes uniformly neutral days and mean-zero polarized days visibly different in the distribution.

In [3]:
SENTIMENT_SIGNED = {"A": -1.0, "B": -0.5, "C": 0.0, "D": 0.5, "E": 1.0}
SENTIMENT_01 = {"A": 0.0, "B": 0.25, "C": 0.5, "D": 0.75, "E": 1.0}
LEVEL_ABC_LMH = {"A": 0.0, "B": 0.5, "C": 1.0, "L": 0.0, "M": 0.5, "H": 1.0}

label_cols = [
    "sentiment_class",
    "time_horizon_class",
    "index_relatability_class",
    "surprise_class",
    "day_uncertainty_class",
    "day_disagreement_class",
]

df = raw.copy()
df["sentiment_signed"] = df["sentiment_class"].map(SENTIMENT_SIGNED)
df["sentiment_01"] = df["sentiment_class"].map(SENTIMENT_01)
df["time_horizon"] = df["time_horizon_class"].map(LEVEL_ABC_LMH)
df["index_relatability"] = df["index_relatability_class"].map(LEVEL_ABC_LMH)
df["surprise"] = df["surprise_class"].map(LEVEL_ABC_LMH)
df["day_uncertainty"] = df["day_uncertainty_class"].map(LEVEL_ABC_LMH)
df["day_disagreement"] = df["day_disagreement_class"].map(LEVEL_ABC_LMH)

malformed = []
maps = {
    "sentiment_class": SENTIMENT_SIGNED,
    "time_horizon_class": LEVEL_ABC_LMH,
    "index_relatability_class": LEVEL_ABC_LMH,
    "surprise_class": LEVEL_ABC_LMH,
    "day_uncertainty_class": LEVEL_ABC_LMH,
    "day_disagreement_class": LEVEL_ABC_LMH,
}
for col, mapping in maps.items():
    bad = ~df[col].isin(mapping.keys())
    malformed.append({"column": col, "malformed_rows": int(bad.sum()), "unique_bad_values": sorted(df.loc[bad, col].dropna().astype(str).unique())})

display(pd.DataFrame(malformed))
display(df[label_cols].apply(lambda s: s.value_counts(dropna=False)).fillna(0).astype(int))

,column,malformed_rows,unique_bad_values
0,sentiment_class,0,[]
1,time_horizon_class,0,[]
2,index_relatability_class,0,[]
3,surprise_class,0,[]
4,day_uncertainty_class,0,[]
5,day_disagreement_class,0,[]


,sentiment_class,time_horizon_class,index_relatability_class,surprise_class,day_uncertainty_class,day_disagreement_class
A,2502,94695,103546,104627,3033,18445
B,24221,30793,26066,34191,66259,80346
C,106592,16762,12555,3404,73051,43552
D,8131,0,0,0,0,0
E,897,0,0,0,0,0
H,0,2,8,0,0,0
L,0,67,160,81,0,0
M,0,24,8,40,0,0


## 3. Build Daily Distributional Features

In [4]:
def fixed_support_entropy(scores: pd.Series, support: Iterable[float] = (-1, -0.5, 0, 0.5, 1)) -> float:
    probs = scores.value_counts(normalize=True).reindex(list(support), fill_value=0.0).values
    probs = probs[probs > 0]
    return float(-(probs * np.log(probs)).sum())


def bimodality_coefficient(scores: pd.Series) -> float:
    x = scores.dropna()
    if len(x) < 4 or x.nunique() < 2:
        return np.nan
    skew = x.skew()
    excess_kurtosis = x.kurt()
    denom = excess_kurtosis + 3
    if denom <= 0:
        return np.nan
    return float((skew ** 2 + 1) / denom)


def classify_regime(row: pd.Series) -> str:
    if row["neutral_mass"] >= 0.75 and row["std_sentiment"] <= 0.25:
        return "neutral_consensus"
    if row["positive_tail_mass"] >= 0.35 and row["negative_tail_mass"] < 0.10:
        return "positive_consensus"
    if row["negative_tail_mass"] >= 0.35 and row["positive_tail_mass"] < 0.10:
        return "negative_consensus"
    if row["bipolarity"] >= 0.20 and row["std_sentiment"] >= 0.35:
        return "polarized"
    if row["negative_tail_mass"] >= 0.15 and row["mean_sentiment"] <= 0.05:
        return "downside_skew"
    if row["positive_tail_mass"] >= 0.15 and row["mean_sentiment"] >= -0.05:
        return "upside_skew"
    return "mixed"


def daily_distribution_features(group: pd.DataFrame) -> pd.Series:
    s = group["sentiment_signed"].dropna()
    pos_tail = float((s >= 0.5).mean()) if len(s) else np.nan
    neg_tail = float((s <= -0.5).mean()) if len(s) else np.nan
    neutral = float((s == 0.0).mean()) if len(s) else np.nan
    return pd.Series({
        "headline_count": len(group),
        "source_count": group["source"].nunique() if "source" in group else np.nan,
        "mean_sentiment": s.mean(),
        "median_sentiment": s.median(),
        "variance_sentiment": s.var(),
        "std_sentiment": s.std(),
        "skew_sentiment": s.skew(),
        "kurtosis_sentiment": s.kurt(),
        "range_sentiment": s.max() - s.min() if len(s) else np.nan,
        "entropy_sentiment": fixed_support_entropy(s),
        "positive_tail_mass": pos_tail,
        "negative_tail_mass": neg_tail,
        "neutral_mass": neutral,
        "extreme_mass": pos_tail + neg_tail,
        "bipolarity": 4 * pos_tail * neg_tail,
        "bimodality_coef": bimodality_coefficient(s),
        "time_horizon_mean": group["time_horizon"].mean(),
        "index_relatability_mean": group["index_relatability"].mean(),
        "surprise_mean": group["surprise"].mean(),
        "day_uncertainty_mean": group["day_uncertainty"].mean(),
        "day_disagreement_mean": group["day_disagreement"].mean(),
    })


daily_feature_cols = [
    "source",
    "sentiment_signed",
    "time_horizon",
    "index_relatability",
    "surprise",
    "day_uncertainty",
    "day_disagreement",
]
daily_input = df.dropna(subset=["Date", "sentiment_signed"])[["Date"] + daily_feature_cols]
daily_sent = (
    daily_input.groupby("Date", sort=True)[daily_feature_cols]
      .apply(daily_distribution_features)
      .reset_index()
)
daily_sent["regime"] = daily_sent.apply(classify_regime, axis=1)

display(daily_sent.head())
display(daily_sent["regime"].value_counts())
display(daily_sent.describe(include="all"))

,Date,headline_count,source_count,mean_sentiment,median_sentiment,variance_sentiment,std_sentiment,skew_sentiment,kurtosis_sentiment,range_sentiment,entropy_sentiment,positive_tail_mass,negative_tail_mass,neutral_mass,extreme_mass,bipolarity,bimodality_coef,time_horizon_mean,index_relatability_mean,surprise_mean,day_uncertainty_mean,day_disagreement_mean,regime
0,2020-05-01,110.0,2.0,-0.136364,0.0,0.105088,0.324172,-0.283928,1.518955,2.0,0.937806,0.054545,0.300000,0.645455,0.354545,0.065455,0.239129,0.227273,0.190909,0.209091,1.0,1.0,downside_skew
1,2020-05-02,54.0,2.0,-0.064815,0.0,0.057041,0.238832,-0.391873,1.139286,1.0,0.681981,0.055556,0.185185,0.759259,0.240741,0.041152,0.278687,0.203704,0.240741,0.092593,1.0,0.5,neutral_consensus
2,2020-05-03,75.0,2.0,-0.126667,0.0,0.068198,0.261148,-0.822310,0.885318,1.5,0.747407,0.026667,0.266667,0.706667,0.293333,0.028444,0.431417,0.253333,0.266667,0.160000,1.0,0.5,downside_skew
3,2020-05-04,114.0,2.0,-0.008772,0.0,0.079568,0.282079,-0.005342,2.439428,2.0,0.822569,0.122807,0.140351,0.736842,0.263158,0.068944,0.183848,0.324561,0.245614,0.140351,1.0,1.0,mixed
4,2020-05-05,112.0,2.0,-0.071429,0.0,0.120978,0.347819,0.200197,1.041998,2.0,1.032609,0.125000,0.267857,0.607143,0.392857,0.133929,0.257318,0.267857,0.258929,0.214286,1.0,1.0,downside_skew


regime
downside_skew         830
neutral_consensus     759
mixed                 141
negative_consensus     74
upside_skew            14
polarized               8
Name: count, dtype: int64

,Date,headline_count,source_count,mean_sentiment,median_sentiment,variance_sentiment,std_sentiment,skew_sentiment,kurtosis_sentiment,range_sentiment,entropy_sentiment,positive_tail_mass,negative_tail_mass,neutral_mass,extreme_mass,bipolarity,bimodality_coef,time_horizon_mean,index_relatability_mean,surprise_mean,day_uncertainty_mean,day_disagreement_mean,regime
count,1826,1826.000000,1826.0,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1824.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826.000000,1826
unique,1826,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6
top,2020-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,downside_skew
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,830
mean,NaN,77.953450,2.0,-0.067589,-0.001095,0.071862,0.260874,-0.443207,3.132896,1.338719,0.709825,0.061990,0.186062,0.751949,0.248051,0.046400,0.358839,0.224551,0.178162,0.140484,0.737824,0.565464,NaN
std,NaN,20.780458,0.0,0.060071,0.023383,0.033104,0.061713,0.999588,3.727416,0.362452,0.197030,0.040313,0.091249,0.100418,0.100418,0.040918,0.188881,0.098348,0.077437,0.076282,0.275762,0.318726,NaN
min,NaN,31.000000,2.0,-0.404762,-0.500000,0.000000,0.000000,-6.782330,-1.845261,0.000000,-0.000000,0.000000,0.000000,0.408163,0.000000,0.000000,0.032787,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,NaN,60.000000,2.0,-0.101813,0.000000,0.048543,0.220325,-0.993851,0.894863,1.000000,0.574996,0.032258,0.120172,0.691021,0.176471,0.017291,0.230539,0.154272,0.123820,0.082115,0.500000,0.500000,NaN
50%,NaN,82.000000,2.0,-0.060440,0.000000,0.067478,0.259766,-0.436656,2.178332,1.500000,0.708717,0.056818,0.173913,0.764706,0.235294,0.036512,0.316160,0.208127,0.166667,0.126893,1.000000,0.500000,NaN
75%,NaN,94.000000,2.0,-0.025674,0.000000,0.090145,0.300242,0.073658,4.191191,1.500000,0.843485,0.085056,0.244186,0.823529,0.308979,0.062500,0.435192,0.280000,0.217391,0.187500,1.000000,1.000000,NaN


## 4. Load Or Fetch Market Data

If `data/market_panel.csv` already exists, this cell uses it. Otherwise set `FETCH_MARKET_DATA = True` and rerun the cell in an environment with network access. The market panel is cached so the empirical cells are reproducible after the first fetch.

In [5]:
def fetch_market_panel() -> pd.DataFrame:
    import yfinance as yf

    spx = yf.download("^SPX", start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
    vix = yf.download("^VIX", start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)

    if isinstance(spx.columns, pd.MultiIndex):
        spx.columns = spx.columns.get_level_values(0)
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = vix.columns.get_level_values(0)

    spx = spx.reset_index()[["Date", "Adj Close", "Volume"]].rename(columns={"Adj Close": "SPX", "Volume": "Volume"})
    vix = vix.reset_index()[["Date", "Adj Close"]].rename(columns={"Adj Close": "VIX"})
    out = pd.merge(spx, vix, on="Date", how="inner")
    out["Date"] = pd.to_datetime(out["Date"]).dt.date
    MARKET_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(MARKET_CACHE_PATH, index=False)
    return out


if MARKET_CACHE_PATH.exists():
    market = pd.read_csv(MARKET_CACHE_PATH)
    market["Date"] = pd.to_datetime(market["Date"]).dt.date
elif FETCH_MARKET_DATA:
    market = fetch_market_panel()
else:
    raise FileNotFoundError(
        f"Missing {MARKET_CACHE_PATH}. Set FETCH_MARKET_DATA=True to fetch SPX/VIX, "
        "or create the cache from a trusted market data source."
    )

market = market.sort_values("Date").reset_index(drop=True)
display(market.head())
display(market.tail())

,Date,SPX,Volume,VIX
0,2020-05-01,2830.709961,4759810000,37.189999
1,2020-05-04,2842.739990,4735930000,35.970001
2,2020-05-05,2868.439941,5140290000,33.610001
3,2020-05-06,2848.419922,4892570000,34.119999
4,2020-05-07,2881.189941,5178790000,31.440001


,Date,SPX,Volume,VIX
1251,2025-04-24,5484.770020,4697710000,26.469999
1252,2025-04-25,5525.209961,4236580000,24.840000
1253,2025-04-28,5528.750000,4257880000,25.150000
1254,2025-04-29,5560.830078,4747150000,24.170000
1255,2025-04-30,5569.060059,5449490000,24.700001


## 5. Create Market Outcomes Without Forward Leakage

Forward outcomes use returns after date `t`. Control variables are lagged or trailing quantities known before the prediction target.

In [6]:
def forward_realized_vol(ret: pd.Series, horizon: int) -> pd.Series:
    future = pd.concat([ret.shift(-i) for i in range(1, horizon + 1)], axis=1)
    return future.std(axis=1) * np.sqrt(252)


market = market.copy()
market["return"] = market["SPX"].pct_change()
market["vix_change"] = market["VIX"].pct_change()
market["volume_log"] = np.log1p(market["Volume"])
market["trailing_rv_5d"] = market["return"].rolling(5).std() * np.sqrt(252)
market["trailing_rv_21d"] = market["return"].rolling(21).std() * np.sqrt(252)
market["fwd_return_1d"] = market["return"].shift(-1)
market["fwd_abs_return_1d"] = market["fwd_return_1d"].abs()
market["fwd_rv_5d"] = forward_realized_vol(market["return"], 5)
market["fwd_abs_return_5d"] = pd.concat([market["return"].shift(-i) for i in range(1, 6)], axis=1).sum(axis=1).abs()

for col in ["return", "VIX", "vix_change", "volume_log", "trailing_rv_5d", "trailing_rv_21d"]:
    market[f"{col}_lag1"] = market[col].shift(1)

tail_cutoff = market["fwd_abs_return_1d"].quantile(TAIL_QUANTILE)
market["tail_1d"] = (market["fwd_abs_return_1d"] >= tail_cutoff).astype(int)

panel = pd.merge(market, daily_sent, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

print(f"Panel rows: {len(panel):,}")
print(f"Date range: {panel['Date'].min()} -> {panel['Date'].max()}")
display(panel.head())
display(panel[["fwd_rv_5d", "tail_1d", "mean_sentiment", "std_sentiment", "bipolarity", "headline_count"]].describe())

Panel rows: 1,256
Date range: 2020-05-01 -> 2025-04-30


,Date,SPX,Volume,VIX,return,vix_change,volume_log,trailing_rv_5d,trailing_rv_21d,fwd_return_1d,fwd_abs_return_1d,fwd_rv_5d,fwd_abs_return_5d,return_lag1,VIX_lag1,vix_change_lag1,volume_log_lag1,trailing_rv_5d_lag1,trailing_rv_21d_lag1,tail_1d,headline_count,source_count,mean_sentiment,median_sentiment,variance_sentiment,std_sentiment,skew_sentiment,kurtosis_sentiment,range_sentiment,entropy_sentiment,positive_tail_mass,negative_tail_mass,neutral_mass,extreme_mass,bipolarity,bimodality_coef,time_horizon_mean,index_relatability_mean,surprise_mean,day_uncertainty_mean,day_disagreement_mean,regime
0,2020-05-01,2830.709961,4759810000,37.189999,NaN,NaN,22.283474,NaN,NaN,0.004250,0.004250,0.143062,0.034687,NaN,NaN,NaN,NaN,NaN,NaN,0,110.0,2.0,-0.136364,0.0,0.105088,0.324172,-0.283928,1.518955,2.0,0.937806,0.054545,0.300000,0.645455,0.354545,0.065455,0.239129,0.227273,0.190909,0.209091,1.0,1.0,downside_skew
1,2020-05-04,2842.739990,4735930000,35.970001,0.004250,-0.032804,22.278444,NaN,NaN,0.009041,0.009041,0.150715,0.030570,NaN,37.189999,NaN,22.283474,NaN,NaN,0,114.0,2.0,-0.008772,0.0,0.079568,0.282079,-0.005342,2.439428,2.0,0.822569,0.122807,0.140351,0.736842,0.263158,0.068944,0.183848,0.324561,0.245614,0.140351,1.0,1.0,mixed
2,2020-05-05,2868.439941,5140290000,33.610001,0.009041,-0.065610,22.360375,NaN,NaN,-0.006979,0.006979,0.236230,0.001030,0.004250,35.970001,-0.032804,22.278444,NaN,NaN,0,112.0,2.0,-0.071429,0.0,0.120978,0.347819,0.200197,1.041998,2.0,1.032609,0.125000,0.267857,0.607143,0.392857,0.133929,0.257318,0.267857,0.258929,0.214286,1.0,1.0,downside_skew
3,2020-05-06,2848.419922,4892570000,34.119999,-0.006979,0.015174,22.310984,NaN,NaN,0.011505,0.011505,0.266148,0.009454,0.009041,33.610001,-0.065610,22.360375,NaN,NaN,0,112.0,2.0,-0.040179,0.0,0.068191,0.261134,-0.492138,1.948684,1.5,0.759757,0.089286,0.160714,0.750000,0.250000,0.057398,0.251016,0.160714,0.183036,0.205357,1.0,1.0,downside_skew
4,2020-05-07,2881.189941,5178790000,31.440001,0.011505,-0.078546,22.367837,NaN,NaN,0.016872,0.016872,0.266212,0.009434,-0.006979,34.119999,0.015174,22.310984,NaN,NaN,0,114.0,2.0,-0.092105,0.0,0.113123,0.336338,-0.119275,2.091332,2.0,0.952873,0.078947,0.245614,0.675439,0.324561,0.077562,0.199207,0.293860,0.162281,0.135965,1.0,1.0,downside_skew


,fwd_rv_5d,tail_1d,mean_sentiment,std_sentiment,bipolarity,headline_count
count,1254.000000,1256.000000,1256.000000,1256.000000,1256.000000,1256.000000
mean,0.154150,0.100318,-0.064618,0.266612,0.049632,89.108280
std,0.094748,0.300544,0.058054,0.058071,0.039845,12.990504
min,0.014343,0.000000,-0.349624,0.000000,0.000000,45.000000
25%,0.092200,0.000000,-0.098653,0.226429,0.020609,80.000000
50%,0.136264,0.000000,-0.057794,0.265914,0.040630,89.000000
75%,0.190052,0.000000,-0.024228,0.302994,0.066149,98.000000
max,0.972949,1.000000,0.102564,0.460690,0.282025,140.000000


## 6. Baseline Versus Distribution Models

In [7]:
import statsmodels.api as sm

CONTROL_COLS = ["return_lag1", "VIX_lag1", "trailing_rv_5d_lag1", "volume_log_lag1"]
BASELINE_FEATURES = CONTROL_COLS + ["mean_sentiment", "headline_count"]
# Keep neutral_mass in the daily table, but omit it from regressions because
# positive_tail_mass + negative_tail_mass + neutral_mass = 1 with a constant.
DISTRIBUTION_FEATURES = [
    "std_sentiment",
    "entropy_sentiment",
    "positive_tail_mass",
    "negative_tail_mass",
    "bipolarity",
    "skew_sentiment",
    "kurtosis_sentiment",
    "range_sentiment",
]


def fit_ols_family(data: pd.DataFrame, target: str, hac_lags: int = 5) -> dict[str, sm.regression.linear_model.RegressionResultsWrapper]:
    needed = [target] + BASELINE_FEATURES + DISTRIBUTION_FEATURES
    sub = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = sub[target]

    X_base = sm.add_constant(sub[BASELINE_FEATURES], has_constant="add")
    X_dist = sm.add_constant(sub[BASELINE_FEATURES + DISTRIBUTION_FEATURES], has_constant="add")

    base = sm.OLS(y, X_base).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    dist = sm.OLS(y, X_dist).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})

    # Non-robust versions only for nested F-test comparison.
    base_nr = sm.OLS(y, X_base).fit()
    dist_nr = sm.OLS(y, X_dist).fit()
    f_stat, f_pvalue, f_df = dist_nr.compare_f_test(base_nr)

    summary = pd.DataFrame([
        {"model": "baseline", "n": int(base.nobs), "adj_r2": base.rsquared_adj, "aic": base.aic, "bic": base.bic},
        {"model": "distribution", "n": int(dist.nobs), "adj_r2": dist.rsquared_adj, "aic": dist.aic, "bic": dist.bic},
    ])
    print(f"Target: {target}")
    display(summary)
    print(f"Nested F-test distribution vs baseline: F={f_stat:.3f}, df={int(f_df)}, p={f_pvalue:.4g}")
    return {"baseline": base, "distribution": dist, "summary": summary}


vol_models = fit_ols_family(panel, PRIMARY_TARGET, hac_lags=5)
display(vol_models["distribution"].summary2().tables[1])

Target: fwd_rv_5d


,model,n,adj_r2,aic,bic
0,baseline,1248,0.305248,-2785.836514,-2749.931432
1,distribution,1248,0.310157,-2786.755870,-2709.816407


Nested F-test distribution vs baseline: F=2.104, df=8, p=0.03275


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
const,-0.053926,0.072509,-0.743710,4.570521e-01,-0.196042,0.088190
return_lag1,-0.602186,0.314819,-1.912800,5.577368e-02,-1.219220,0.014848
VIX_lag1,0.006936,0.000982,7.059657,1.669147e-12,0.005010,0.008862
trailing_rv_5d_lag1,0.062053,0.054174,1.145457,2.520198e-01,-0.044125,0.168232
volume_log_lag1,0.002372,0.002708,0.875999,3.810305e-01,-0.002935,0.007679
mean_sentiment,-0.568273,0.398472,-1.426129,1.538310e-01,-1.349264,0.212718
headline_count,-0.000377,0.000254,-1.485942,1.372945e-01,-0.000875,0.000120
std_sentiment,-0.028865,0.200971,-0.143629,8.857933e-01,-0.422762,0.365031
entropy_sentiment,0.058871,0.112353,0.523981,6.002921e-01,-0.161337,0.279078
positive_tail_mass,0.201923,0.430241,0.469325,6.388373e-01,-0.641334,1.045180


## 7. Tail-Risk Logit

In [8]:
import statsmodels.api as sm
from scipy.stats import chi2


def fit_logit_family(data: pd.DataFrame, target: str = "tail_1d"):
    needed = [target] + BASELINE_FEATURES + DISTRIBUTION_FEATURES
    sub = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = sub[target]
    X_base = sm.add_constant(sub[BASELINE_FEATURES], has_constant="add")
    X_dist = sm.add_constant(sub[BASELINE_FEATURES + DISTRIBUTION_FEATURES], has_constant="add")

    base = sm.Logit(y, X_base).fit(disp=False)
    dist = sm.Logit(y, X_dist).fit(disp=False)

    lr_stat = 2 * (dist.llf - base.llf)
    df_diff = dist.df_model - base.df_model
    lr_pvalue = float(chi2.sf(lr_stat, df_diff))

    summary = pd.DataFrame([
        {"model": "baseline", "n": int(base.nobs), "pseudo_r2": base.prsquared, "aic": base.aic, "bic": base.bic},
        {"model": "distribution", "n": int(dist.nobs), "pseudo_r2": dist.prsquared, "aic": dist.aic, "bic": dist.bic},
    ])
    print(f"Target: {target}")
    display(summary)
    print(f"LR test distribution vs baseline: chi2={lr_stat:.3f}, df={int(df_diff)}, p={lr_pvalue:.4g}")
    return {"baseline": base, "distribution": dist, "summary": summary}


tail_models = fit_logit_family(panel)
display(tail_models["distribution"].summary2().tables[1])
display(np.exp(tail_models["distribution"].params).rename("odds_ratio"))

Target: tail_1d


,model,n,pseudo_r2,aic,bic
0,baseline,1250,0.170503,691.776209,727.692500
1,distribution,1250,0.179445,700.469674,777.433157


LR test distribution vs baseline: chi2=7.307, df=8, p=0.5039


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
const,-12.803809,12.948984,-0.988789,3.227665e-01,-38.183352,12.575733
return_lag1,-2.057841,7.398790,-0.278132,7.809110e-01,-16.559203,12.443521
VIX_lag1,0.156164,0.025919,6.025060,1.690473e-09,0.105364,0.206965
trailing_rv_5d_lag1,0.334994,1.164785,0.287601,7.736518e-01,-1.947943,2.617930
volume_log_lag1,0.314643,0.592539,0.531009,5.954128e-01,-0.846711,1.475997
mean_sentiment,-11.998817,11.570025,-1.037061,2.997076e-01,-34.675651,10.678016
headline_count,-0.000477,0.009049,-0.052766,9.579187e-01,-0.018212,0.017258
std_sentiment,-14.440817,9.765359,-1.478780,1.391991e-01,-33.580569,4.698934
entropy_sentiment,6.454297,5.305166,1.216606,2.237541e-01,-3.943637,16.852230
positive_tail_mass,-5.061950,14.294954,-0.354107,7.232583e-01,-33.079544,22.955644


const                  2.750275e-06
return_lag1            1.277295e-01
VIX_lag1               1.169018e+00
trailing_rv_5d_lag1    1.397932e+00
volume_log_lag1        1.369770e+00
mean_sentiment         6.151483e-06
headline_count         9.995227e-01
std_sentiment          5.350972e-07
entropy_sentiment      6.354266e+02
positive_tail_mass     6.333198e-03
negative_tail_mass     4.731674e-04
bipolarity             2.567084e-03
skew_sentiment         1.229784e+00
kurtosis_sentiment     1.000743e+00
range_sentiment        1.354230e+00
Name: odds_ratio, dtype: float64

## 8. Regime Model

This tests the paper intuition directly: same or similar means may correspond to different distributional states.

In [9]:
import statsmodels.api as sm


def fit_regime_model(data: pd.DataFrame, target: str = PRIMARY_TARGET, hac_lags: int = 5):
    regime_dummies = pd.get_dummies(data["regime"], prefix="regime", drop_first=True, dtype=float)
    work = pd.concat([data.reset_index(drop=True), regime_dummies.reset_index(drop=True)], axis=1)
    regime_cols = list(regime_dummies.columns)
    features = BASELINE_FEATURES + regime_cols
    sub = work[[target] + features].replace([np.inf, -np.inf], np.nan).dropna()
    y = sub[target]
    X = sm.add_constant(sub[features], has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    return model, regime_cols


regime_model, regime_cols = fit_regime_model(panel)
print(regime_cols)
display(regime_model.summary2().tables[1])

['regime_mixed', 'regime_negative_consensus', 'regime_neutral_consensus', 'regime_polarized', 'regime_upside_skew']


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
const,-0.044654,0.058028,-0.769532,4.415774e-01,-0.158386,0.069078
return_lag1,-0.621507,0.327351,-1.898596,5.761763e-02,-1.263103,0.020089
VIX_lag1,0.006975,0.000974,7.159690,8.085995e-13,0.005065,0.008884
trailing_rv_5d_lag1,0.069756,0.055115,1.265642,2.056414e-01,-0.038268,0.177780
volume_log_lag1,0.002202,0.002600,0.846945,3.970257e-01,-0.002894,0.007299
mean_sentiment,-0.331721,0.082917,-4.000644,6.317031e-05,-0.494235,-0.169207
headline_count,-0.000362,0.000269,-1.347078,1.779552e-01,-0.000888,0.000165
regime_mixed,0.018787,0.008827,2.128286,3.331340e-02,0.001486,0.036088
regime_negative_consensus,-0.001023,0.030457,-0.033597,9.731984e-01,-0.060717,0.058671
regime_neutral_consensus,0.012533,0.005351,2.342191,1.917090e-02,0.002045,0.023020


## 9. Expanding-Window Out-Of-Sample Check

In [10]:
import statsmodels.api as sm


def expanding_oos(data: pd.DataFrame, target: str, features: list[str], split_date: str) -> pd.DataFrame:
    needed = ["Date", target] + features
    sub = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    sub["Date"] = pd.to_datetime(sub["Date"])
    sub = sub.sort_values("Date").reset_index(drop=True)
    split = pd.Timestamp(split_date)

    rows = []
    for i, row in sub[sub["Date"] >= split].iterrows():
        train = sub.loc[sub.index < i]
        if len(train) < max(120, len(features) * 10):
            continue
        X_train = sm.add_constant(train[features], has_constant="add")
        y_train = train[target]
        model = sm.OLS(y_train, X_train).fit()
        X_test = sm.add_constant(row[features].to_frame().T, has_constant="add").reindex(columns=X_train.columns, fill_value=0)
        rows.append({"Date": row["Date"], "actual": row[target], "pred": float(model.predict(X_test).iloc[0])})
    return pd.DataFrame(rows)


def oos_metrics(oos: pd.DataFrame) -> dict[str, float]:
    rmse = float(np.sqrt(np.mean((oos["actual"] - oos["pred"]) ** 2)))
    ss_res = ((oos["actual"] - oos["pred"]) ** 2).sum()
    ss_tot = ((oos["actual"] - oos["actual"].mean()) ** 2).sum()
    return {"n": len(oos), "rmse": rmse, "oos_r2": 1 - ss_res / ss_tot}


oos_base = expanding_oos(panel, PRIMARY_TARGET, BASELINE_FEATURES, OOS_SPLIT_DATE)
oos_dist = expanding_oos(panel, PRIMARY_TARGET, BASELINE_FEATURES + DISTRIBUTION_FEATURES, OOS_SPLIT_DATE)

oos_summary = pd.DataFrame([
    {"model": "baseline", **oos_metrics(oos_base)},
    {"model": "distribution", **oos_metrics(oos_dist)},
])
display(oos_summary)

oos_compare = oos_base.rename(columns={"pred": "pred_base"}).merge(
    oos_dist.rename(columns={"pred": "pred_dist"}), on=["Date", "actual"], how="inner"
)
oos_compare["sq_err_base"] = (oos_compare["actual"] - oos_compare["pred_base"]) ** 2
oos_compare["sq_err_dist"] = (oos_compare["actual"] - oos_compare["pred_dist"]) ** 2
print("Distribution model wins on %.1f%% of OOS days" % (100 * (oos_compare["sq_err_dist"] < oos_compare["sq_err_base"]).mean()))
display(oos_compare.tail())

,model,n,rmse,oos_r2
0,baseline,581,0.079017,0.257235
1,distribution,581,0.079745,0.243494


Distribution model wins on 49.2% of OOS days


,Date,actual,pred_base,pred_dist,sq_err_base,sq_err_dist
576,2025-04-22,0.128384,0.256497,0.249691,0.016413,0.014715
577,2025-04-23,0.125053,0.230756,0.227784,0.011173,0.010554
578,2025-04-24,0.051958,0.235732,0.226586,0.033773,0.030495
579,2025-04-25,0.043969,0.195918,0.207736,0.023088,0.026820
580,2025-04-28,0.048519,0.190764,0.191245,0.020234,0.020371


## 10. Compact Primary Model And Robustness

The broad distribution model is intentionally exploratory and contains many correlated shape features. This section defines a smaller primary distribution model and checks whether its incremental value survives source splits, high-relevance filters, and date-shuffle placebo tests.

In [11]:
PRIMARY_DISTRIBUTION_FEATURES = [
    "negative_tail_mass",
    "positive_tail_mass",
    "entropy_sentiment",
    "bipolarity",
]


def build_daily_panel_from_headlines(headlines: pd.DataFrame, label: str) -> pd.DataFrame:
    needed = ["Date"] + daily_feature_cols
    clean = headlines.dropna(subset=["Date", "sentiment_signed"])[needed].copy()
    daily = (
        clean.groupby("Date", sort=True)[daily_feature_cols]
        .apply(daily_distribution_features)
        .reset_index()
    )
    daily["regime"] = daily.apply(classify_regime, axis=1)
    out = pd.merge(market, daily, on="Date", how="inner").sort_values("Date").reset_index(drop=True)
    out["sample"] = label
    return out


def fit_compact_ols(data: pd.DataFrame, target: str = PRIMARY_TARGET, features: list[str] = PRIMARY_DISTRIBUTION_FEATURES, hac_lags: int = 5) -> dict:
    needed = [target] + BASELINE_FEATURES + features
    sub = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    y = sub[target]
    X_base = sm.add_constant(sub[BASELINE_FEATURES], has_constant="add")
    X_dist = sm.add_constant(sub[BASELINE_FEATURES + features], has_constant="add")

    base = sm.OLS(y, X_base).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    dist = sm.OLS(y, X_dist).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})

    base_nr = sm.OLS(y, X_base).fit()
    dist_nr = sm.OLS(y, X_dist).fit()
    f_stat, f_pvalue, f_df = dist_nr.compare_f_test(base_nr)

    return {
        "n": int(dist.nobs),
        "base_adj_r2": base.rsquared_adj,
        "dist_adj_r2": dist.rsquared_adj,
        "delta_adj_r2": dist.rsquared_adj - base.rsquared_adj,
        "base_aic": base.aic,
        "dist_aic": dist.aic,
        "delta_aic": dist.aic - base.aic,
        "base_bic": base.bic,
        "dist_bic": dist.bic,
        "delta_bic": dist.bic - base.bic,
        "f_stat": float(f_stat),
        "f_pvalue": float(f_pvalue),
        "f_df": int(f_df),
        "base_model": base,
        "dist_model": dist,
    }


compact_all = fit_compact_ols(panel)
compact_summary = pd.DataFrame([{k: v for k, v in compact_all.items() if not k.endswith("model")}])
display(compact_summary)
display(compact_all["dist_model"].summary2().tables[1])

,n,base_adj_r2,dist_adj_r2,delta_adj_r2,base_aic,dist_aic,delta_aic,base_bic,dist_bic,delta_bic,f_stat,f_pvalue,f_df
0,1248,0.305248,0.311683,0.006434,-2785.836514,-2793.477748,-7.641233,-2749.931432,-2737.055475,12.875957,3.900232,0.003743,4


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
const,-0.037040,0.062755,-0.590233,5.550344e-01,-0.160038,0.085957
return_lag1,-0.583515,0.307011,-1.900632,5.735020e-02,-1.185245,0.018215
VIX_lag1,0.006960,0.000982,7.085964,1.380792e-12,0.005035,0.008886
trailing_rv_5d_lag1,0.062126,0.054162,1.147043,2.513639e-01,-0.044029,0.168282
volume_log_lag1,0.002168,0.002610,0.830689,4.061495e-01,-0.002948,0.007284
mean_sentiment,-0.562417,0.311854,-1.803462,7.131569e-02,-1.173639,0.048806
headline_count,-0.000367,0.000253,-1.450847,1.468225e-01,-0.000862,0.000129
negative_tail_mass,-0.210941,0.136404,-1.546449,1.219962e-01,-0.478287,0.056405
positive_tail_mass,0.178162,0.384276,0.463631,6.429123e-01,-0.575005,0.931329
entropy_sentiment,0.037683,0.044493,0.846943,3.970271e-01,-0.049521,0.124887


### Source And Relevance Robustness

Run the compact model on all headlines, each source separately, and higher-index-relatability subsets. The high-relevance split is important because the raw corpus contains broad news, not only market-moving financial news.

In [12]:
sample_definitions = {
    "all": df,
    "nytimes_only": df[df["source"].eq("NYTimes")],
    "guardian_only": df[df["source"].eq("Guardian")],
    "high_relatability_C": df[df["index_relatability"].ge(1.0)],
    "medium_high_relatability_BC": df[df["index_relatability"].ge(0.5)],
}

robust_rows = []
robust_panels = {}
for label, sample_df in sample_definitions.items():
    sample_panel = build_daily_panel_from_headlines(sample_df, label)
    robust_panels[label] = sample_panel
    try:
        fit = fit_compact_ols(sample_panel)
        row = {k: v for k, v in fit.items() if not k.endswith("model")}
        row.update({
            "sample": label,
            "headline_rows": len(sample_df),
            "market_days": len(sample_panel),
            "mean_headline_count": sample_panel["headline_count"].mean(),
        })
        robust_rows.append(row)
    except Exception as exc:
        robust_rows.append({"sample": label, "headline_rows": len(sample_df), "market_days": len(sample_panel), "error": str(exc)})

robust_summary = pd.DataFrame(robust_rows)
front_cols = ["sample", "headline_rows", "market_days", "mean_headline_count", "n"]
other_cols = [c for c in robust_summary.columns if c not in front_cols]
display(robust_summary[front_cols + other_cols].sort_values("sample"))

,sample,headline_rows,market_days,mean_headline_count,n,base_adj_r2,dist_adj_r2,delta_adj_r2,base_aic,dist_aic,delta_aic,base_bic,dist_bic,delta_bic,f_stat,f_pvalue,f_df
0,all,142343,1256,89.108280,1248,0.305248,0.311683,0.006434,-2785.836514,-2793.477748,-7.641233,-2749.931432,-2737.055475,12.875957,3.900232,0.003743,4
2,guardian_only,90584,1256,56.091561,1248,0.296896,0.305775,0.008879,-2770.921509,-2782.811470,-11.889961,-2735.016427,-2726.389197,8.627229,4.968147,0.000564,4
3,high_relatability_C,12563,1238,7.908724,1230,0.295425,0.297363,0.001938,-2718.803373,-2718.220725,0.582648,-2682.999987,-2661.958261,21.041726,1.843307,0.118181,4
4,medium_high_relatability_BC,38637,1256,24.220541,1248,0.301005,0.300307,-0.000698,-2778.237141,-2773.019853,5.217288,-2742.332058,-2716.597580,25.734478,0.690316,0.598685,4
1,nytimes_only,51759,1256,33.016720,1248,0.303498,0.308086,0.004589,-2782.695452,-2786.973484,-4.278032,-2746.790369,-2730.551211,16.239158,3.057468,0.016064,4


### Compact Model OOS Robustness

The strongest claim would require the compact distribution model to improve out-of-sample volatility forecasts, not just in-sample fit.

In [13]:
def compare_oos_for_features(data: pd.DataFrame, target: str, dist_features: list[str], split_date: str = OOS_SPLIT_DATE) -> dict:
    base = expanding_oos(data, target, BASELINE_FEATURES, split_date)
    dist = expanding_oos(data, target, BASELINE_FEATURES + dist_features, split_date)
    merged = base.rename(columns={"pred": "pred_base"}).merge(
        dist.rename(columns={"pred": "pred_dist"}), on=["Date", "actual"], how="inner"
    )
    merged["sq_err_base"] = (merged["actual"] - merged["pred_base"]) ** 2
    merged["sq_err_dist"] = (merged["actual"] - merged["pred_dist"]) ** 2
    return {
        "n": len(merged),
        "base_rmse": oos_metrics(base)["rmse"],
        "dist_rmse": oos_metrics(dist)["rmse"],
        "base_oos_r2": oos_metrics(base)["oos_r2"],
        "dist_oos_r2": oos_metrics(dist)["oos_r2"],
        "delta_oos_r2": oos_metrics(dist)["oos_r2"] - oos_metrics(base)["oos_r2"],
        "dist_win_rate": float((merged["sq_err_dist"] < merged["sq_err_base"]).mean()),
    }


oos_rows = []
for label, sample_panel in robust_panels.items():
    try:
        row = compare_oos_for_features(sample_panel, PRIMARY_TARGET, PRIMARY_DISTRIBUTION_FEATURES)
        row["sample"] = label
        oos_rows.append(row)
    except Exception as exc:
        oos_rows.append({"sample": label, "error": str(exc)})

oos_robust_summary = pd.DataFrame(oos_rows)
display(oos_robust_summary[["sample"] + [c for c in oos_robust_summary.columns if c != "sample"]].sort_values("sample"))

,sample,n,base_rmse,dist_rmse,base_oos_r2,dist_oos_r2,delta_oos_r2,dist_win_rate
0,all,581,0.079017,0.079264,0.257235,0.252588,-0.004647,0.507745
2,guardian_only,581,0.079541,0.079146,0.247360,0.254814,0.007454,0.506024
3,high_relatability_C,564,0.080323,0.080127,0.245564,0.249239,0.003675,0.485816
4,medium_high_relatability_BC,581,0.080204,0.080346,0.234758,0.232041,-0.002717,0.450947
1,nytimes_only,581,0.079874,0.079249,0.241033,0.252863,0.011831,0.481928


### Date-Shuffle Placebo

This keeps market outcomes fixed but randomly permutes the compact distribution features across dates. The real incremental F-statistic should sit in the upper tail of the placebo distribution if timing contains genuine information.

In [14]:
def shuffle_distribution_features(data: pd.DataFrame, features: list[str], seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    shuffled = data.copy()
    perm = rng.permutation(len(shuffled))
    shuffled.loc[:, features] = shuffled[features].to_numpy()[perm]
    return shuffled


N_PLACEBO = 250
real_fit = fit_compact_ols(panel)
placebo_rows = []
for seed in range(N_PLACEBO):
    shuffled_panel = shuffle_distribution_features(panel, PRIMARY_DISTRIBUTION_FEATURES, seed)
    fit = fit_compact_ols(shuffled_panel)
    placebo_rows.append({
        "seed": seed,
        "f_stat": fit["f_stat"],
        "f_pvalue": fit["f_pvalue"],
        "delta_adj_r2": fit["delta_adj_r2"],
        "delta_aic": fit["delta_aic"],
    })

placebo = pd.DataFrame(placebo_rows)
empirical_p_f = float((placebo["f_stat"] >= real_fit["f_stat"]).mean())
empirical_p_r2 = float((placebo["delta_adj_r2"] >= real_fit["delta_adj_r2"]).mean())

print(f"Real compact F-stat: {real_fit['f_stat']:.3f}; parametric p={real_fit['f_pvalue']:.4g}; placebo empirical p={empirical_p_f:.4f}")
print(f"Real compact delta adj-R2: {real_fit['delta_adj_r2']:.5f}; placebo empirical p={empirical_p_r2:.4f}")
display(placebo.describe())

Real compact F-stat: 3.900; parametric p=0.003743; placebo empirical p=0.0040
Real compact delta adj-R2: 0.00643; placebo empirical p=0.0040


,seed,f_stat,f_pvalue,delta_adj_r2,delta_aic
count,250.000000,250.000000,250.000000,250.000000,250.000000
mean,124.500000,0.983415,0.508614,-0.000041,4.040872
std,72.312977,0.706199,0.288195,0.001576,2.836292
min,0.000000,0.021055,0.001434,-0.002199,-9.808570
25%,62.250000,0.486435,0.261903,-0.001152,2.700220
50%,124.500000,0.824150,0.509752,-0.000394,4.678510
75%,186.750000,1.316059,0.745730,0.000707,6.038499
max,249.000000,4.444537,0.999137,0.007629,7.915035


## 11. Categorical Sentiment-Vector Features

The signed-score moments above are useful, but they still treat ordered classes as if they were continuous measurements. This section keeps the LLM output as a categorical distribution: each day is represented by its class-share vector over A/B/C/D/E plus interpretable transforms. This is closer to the multi-axis discrete-vector idea: keep discrete judgments, then test which axes matter.

In [ ]:
CLASS_ORDER = ["A", "B", "C", "D", "E"]


def categorical_sentiment_features(group: pd.DataFrame) -> pd.Series:
    shares = group["sentiment_class"].value_counts(normalize=True).reindex(CLASS_ORDER, fill_value=0.0)
    p_a, p_b, p_c, p_d, p_e = [float(shares[c]) for c in CLASS_ORDER]
    neg_mass = p_a + p_b
    pos_mass = p_d + p_e
    nonneutral_mass = 1.0 - p_c
    probs = shares.values[shares.values > 0]
    class_entropy = float(-(probs * np.log(probs)).sum())
    max_share = float(shares.max())
    return pd.Series({
        "share_A_strong_neg": p_a,
        "share_B_mod_neg": p_b,
        "share_C_neutral": p_c,
        "share_D_mod_pos": p_d,
        "share_E_strong_pos": p_e,
        "cat_neg_mass": neg_mass,
        "cat_pos_mass": pos_mass,
        "cat_net_sentiment": pos_mass - neg_mass,
        "cat_neutral_mass": p_c,
        "cat_nonneutral_mass": nonneutral_mass,
        "cat_entropy": class_entropy,
        "cat_max_share": max_share,
        "cat_disagreement": 1.0 - max_share,
        "cat_polarization": 4.0 * neg_mass * pos_mass,
        "cat_strong_tail_mass": p_a + p_e,
        "cat_downside_warning": p_a + p_b * p_c,
        "cat_tail_imbalance": p_a - p_e,
    })


def build_categorical_panel(headlines: pd.DataFrame, label: str) -> pd.DataFrame:
    daily_cat = (
        headlines.dropna(subset=["Date", "sentiment_class"])
        .groupby("Date", sort=True)
        .apply(categorical_sentiment_features)
        .reset_index()
    )
    base_panel = build_daily_panel_from_headlines(headlines, label)
    out = pd.merge(base_panel, daily_cat, on="Date", how="left")
    return out


categorical_panel = build_categorical_panel(df, "all")
display(categorical_panel[[
    "Date", "headline_count", "mean_sentiment", "cat_neg_mass", "cat_pos_mass",
    "cat_neutral_mass", "cat_entropy", "cat_disagreement", "cat_polarization",
    "cat_downside_warning", "cat_tail_imbalance"
]].head())
display(categorical_panel[[
    "cat_neg_mass", "cat_pos_mass", "cat_neutral_mass", "cat_entropy",
    "cat_disagreement", "cat_polarization", "cat_downside_warning", "cat_tail_imbalance"
]].describe())

### Categorical Vector Model

This model is intentionally compact. It avoids using all five class shares at once because they sum to one. The main axes are: negative mass, positive mass, entropy/disagreement, polarization, and downside-warning imbalance.

In [ ]:
CATEGORICAL_PRIMARY_FEATURES = [
    "cat_neg_mass",
    "cat_pos_mass",
    "cat_entropy",
    "cat_polarization",
    "cat_downside_warning",
    "cat_tail_imbalance",
]


categorical_fit = fit_compact_ols(categorical_panel, features=CATEGORICAL_PRIMARY_FEATURES)
categorical_summary = pd.DataFrame([{k: v for k, v in categorical_fit.items() if not k.endswith("model")}])
display(categorical_summary)
display(categorical_fit["dist_model"].summary2().tables[1])

categorical_oos = compare_oos_for_features(categorical_panel, PRIMARY_TARGET, CATEGORICAL_PRIMARY_FEATURES)
display(pd.DataFrame([categorical_oos]))

### Categorical Source And Relevance Robustness

Run the same categorical-vector model across the same source and relevance splits used above.

In [ ]:
cat_robust_rows = []
cat_robust_panels = {}
for label, sample_df in sample_definitions.items():
    sample_panel = build_categorical_panel(sample_df, label)
    cat_robust_panels[label] = sample_panel
    try:
        fit = fit_compact_ols(sample_panel, features=CATEGORICAL_PRIMARY_FEATURES)
        oos = compare_oos_for_features(sample_panel, PRIMARY_TARGET, CATEGORICAL_PRIMARY_FEATURES)
        row = {k: v for k, v in fit.items() if not k.endswith("model")}
        row.update({f"oos_{k}": v for k, v in oos.items()})
        row.update({
            "sample": label,
            "headline_rows": len(sample_df),
            "market_days": len(sample_panel),
            "mean_headline_count": sample_panel["headline_count"].mean(),
        })
        cat_robust_rows.append(row)
    except Exception as exc:
        cat_robust_rows.append({"sample": label, "headline_rows": len(sample_df), "market_days": len(sample_panel), "error": str(exc)})

cat_robust_summary = pd.DataFrame(cat_robust_rows)
front_cols = ["sample", "headline_rows", "market_days", "mean_headline_count", "n", "delta_adj_r2", "f_pvalue", "delta_aic", "delta_bic", "oos_delta_oos_r2", "oos_dist_win_rate"]
display(cat_robust_summary[[c for c in front_cols if c in cat_robust_summary.columns]].sort_values("sample"))

### Categorical Date-Shuffle Placebo

This repeats the date-shuffle placebo for the categorical-vector axes.

In [ ]:
N_CAT_PLACEBO = 250
real_cat_fit = fit_compact_ols(categorical_panel, features=CATEGORICAL_PRIMARY_FEATURES)
cat_placebo_rows = []
for seed in range(N_CAT_PLACEBO):
    shuffled_panel = shuffle_distribution_features(categorical_panel, CATEGORICAL_PRIMARY_FEATURES, seed)
    fit = fit_compact_ols(shuffled_panel, features=CATEGORICAL_PRIMARY_FEATURES)
    cat_placebo_rows.append({
        "seed": seed,
        "f_stat": fit["f_stat"],
        "f_pvalue": fit["f_pvalue"],
        "delta_adj_r2": fit["delta_adj_r2"],
        "delta_aic": fit["delta_aic"],
    })

cat_placebo = pd.DataFrame(cat_placebo_rows)
cat_empirical_p_f = float((cat_placebo["f_stat"] >= real_cat_fit["f_stat"]).mean())
cat_empirical_p_r2 = float((cat_placebo["delta_adj_r2"] >= real_cat_fit["delta_adj_r2"]).mean())

print(f"Real categorical F-stat: {real_cat_fit['f_stat']:.3f}; parametric p={real_cat_fit['f_pvalue']:.4g}; placebo empirical p={cat_empirical_p_f:.4f}")
print(f"Real categorical delta adj-R2: {real_cat_fit['delta_adj_r2']:.5f}; placebo empirical p={cat_empirical_p_r2:.4f}")
display(cat_placebo.describe())

## 12. Second Student Intraday Data Readiness

The second student's data is now expected under `data/`. This section only checks availability, schemas, and date ranges. The actual intraday analysis should be ported carefully after fixing the timing issue: market reaction windows must start after the actual headline timestamp, not at the floored bucket start.

In [15]:
INTRADAY_DATA_DIR = Path("data")
INTRADAY_NEWS_PATH = INTRADAY_DATA_DIR / "news_data.csv"
INTRADAY_DAILY_PATH = INTRADAY_DATA_DIR / "news_daily_data"
SPY_1MIN_PATH = INTRADAY_DATA_DIR / "spy_1min_data"

intraday_paths = {
    "news_data": INTRADAY_NEWS_PATH,
    "news_daily_data": INTRADAY_DAILY_PATH,
    "spy_1min_data": SPY_1MIN_PATH,
}

path_audit = pd.DataFrame([
    {"name": name, "path": str(path), "exists": path.exists(), "size_mb": path.stat().st_size / 1_000_000 if path.exists() else np.nan}
    for name, path in intraday_paths.items()
])
display(path_audit)

missing = path_audit.loc[~path_audit["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(f"Missing intraday files: {missing}")

intraday_news = pd.read_csv(INTRADAY_NEWS_PATH, low_memory=False)
intraday_daily = pd.read_csv(INTRADAY_DAILY_PATH, low_memory=False)
spy_1min = pd.read_csv(SPY_1MIN_PATH, low_memory=False)

intraday_news["date"] = pd.to_datetime(intraday_news["date"], utc=True, errors="coerce")
intraday_daily["date"] = pd.to_datetime(intraday_daily["date"], errors="coerce")
spy_1min["date"] = pd.to_datetime(spy_1min["date"], errors="coerce")

intraday_audit = pd.DataFrame([
    {
        "dataset": "intraday_news",
        "rows": len(intraday_news),
        "columns": len(intraday_news.columns),
        "start": intraday_news["date"].min(),
        "end": intraday_news["date"].max(),
    },
    {
        "dataset": "intraday_daily",
        "rows": len(intraday_daily),
        "columns": len(intraday_daily.columns),
        "start": intraday_daily["date"].min(),
        "end": intraday_daily["date"].max(),
    },
    {
        "dataset": "spy_1min",
        "rows": len(spy_1min),
        "columns": len(spy_1min.columns),
        "start": spy_1min["date"].min(),
        "end": spy_1min["date"].max(),
    },
])
display(intraday_audit)

print("intraday_news columns:", list(intraday_news.columns))
print("intraday_daily columns:", list(intraday_daily.columns))
print("spy_1min columns:", list(spy_1min.columns))
display(intraday_news.head(3))
display(intraday_daily.head(3))
display(spy_1min.head(3))

,name,path,exists,size_mb
0,news_data,data/news_data.csv,True,21.291011
1,news_daily_data,data/news_daily_data,True,0.056531
2,spy_1min_data,data/spy_1min_data,True,36.448727


,dataset,rows,columns,start,end
0,intraday_news,142343,13,2020-05-01 00:00:09+00:00,2025-04-30 22:59:23+00:00
1,intraday_daily,1243,14,2020-05-20 00:00:00,2025-04-30 00:00:00
2,spy_1min,595210,7,2020-03-24 10:07:00,2026-05-01 15:59:00


intraday_news columns: ['headline', 'source', 'date', 'content', 'section_name', 'event', 'sentiment_class', 'time_horizon_class', 'index_relatability_class', 'surprise_class', 'day_uncertainty_class', 'day_disagreement_class', 'Date']
intraday_daily columns: ['Unnamed: 0', 'date', 'unc_nyt', 'sent_nyt', 'unc_guard', 'sent_guard', 'unc_nyt_world', 'sent_nyt_world', 'unc_nyt_business', 'sent_nyt_business', 'unc_guard_world', 'sent_guard_world', 'unc_guard_business', 'sent_guard_business']
spy_1min columns: ['date', 'open', 'high', 'low', 'close', 'change', 'volume']


,headline,source,date,content,section_name,event,sentiment_class,time_horizon_class,index_relatability_class,surprise_class,day_uncertainty_class,day_disagreement_class,Date
0,U.S. Stocks End the Week Lower After Tech Earn...,NYTimes,2020-05-01 04:01:12+00:00,NaN,Business Day,Overall Dataset,B,A,C,A,C,C,2020-05-01
1,Companies Sell the Blood of Recovered Coronavi...,NYTimes,2020-05-01 04:01:49+00:00,NaN,World,Overall Dataset,B,B,A,B,C,C,2020-05-01
2,Hundreds of Rohingya Refugees Stuck at Sea Wit...,NYTimes,2020-05-01 04:39:21+00:00,NaN,World,Overall Dataset,C,A,A,A,C,C,2020-05-01


,Unnamed: 0,date,unc_nyt,sent_nyt,unc_guard,sent_guard,unc_nyt_world,sent_nyt_world,unc_nyt_business,sent_nyt_business,unc_guard_world,sent_guard_world,unc_guard_business,sent_guard_business
0,0,2020-05-20,3.0,0.0,NaN,NaN,3.0,0.0,4.0,-1.0,NaN,NaN,NaN,NaN
1,1,2020-05-21,3.0,-1.0,NaN,NaN,3.0,0.0,5.0,-2.0,NaN,NaN,NaN,NaN
2,2,2020-05-22,3.0,0.0,NaN,NaN,3.0,0.0,3.0,0.0,NaN,NaN,NaN,NaN


,date,open,high,low,close,change,volume
0,2020-03-24 10:07:00,236.09,236.50,236.08,236.20,0.05,359261.0
1,2020-03-24 10:08:00,236.20,236.37,235.60,235.77,-0.18,480055.0
2,2020-03-24 10:09:00,235.80,236.07,235.54,235.82,0.01,958555.0


## 13. Placebo And Robustness TODOs

These are intentionally listed here before exploratory interpretation:

- Shuffle dates and rerun the baseline versus distribution model.
- Shuffle sentiment labels within source/year.
- Run NYTimes-only, Guardian-only, and both-source samples.
- Compare all headlines against high-index-relatability headlines.
- Compare signed sentiment against original `0..1` mapping.
- Later: compare against continuous LLM scores or FinBERT probabilities.

Do not treat a result as paper-ready until it survives at least the source and timing robustness checks.